# MeterMind — Dataset Generation

Generates **(expanded archaic prose, original poetic line)** pairs for GRU seq2seq training.

Each Shakespeare line gets **3 different expansions**, giving the GRU multiple prose inputs that all map to the same metered target.

**Expansion rule:** All original words must appear verbatim in the expansion. Archaic register (thee, thou, doth) must be preserved. New words may be added to make it read naturally.

```
shakespeare_sonnets_clean.txt → LLM expansion (×3) → validation → training_pairs.csv
```

## 0. Install dependencies

In [ ]:
%pip install anthropic pandas tqdm --quiet

## 1. Configuration

In [ ]:
import anthropic, json, re, time, os
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

# ── API ───────────────────────────────────────────────────────────────────────
# Set ANTHROPIC_API_KEY in your environment, or paste it directly here.
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", "YOUR_KEY_HERE"))

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL      = "claude-haiku-4-5-20251001"   # cheap & fast; swap for claude-sonnet-4-6 for higher quality
MAX_TOKENS = 600                            # enough for 3 expansions per call

# ── Corpus ────────────────────────────────────────────────────────────────────
CORPUS_PATH = "shakespeare_sonnets_clean.txt"   # ← one line per row, elisions already expanded

# ── Generation ────────────────────────────────────────────────────────────────
N_EXPANSIONS  = 3      # expansions per line
RETRY_LIMIT   = 3      # retries on API or validation failure
SLEEP_BETWEEN = 0.3    # seconds between calls (keeps you under 30 RPM)

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_CSV    = "training_pairs.csv"
OUTPUT_JSONL  = "training_pairs.jsonl"
CHECKPOINT    = "checkpoint.json"    # saves after every line so you can resume
FAILURES_LOG  = "failures.json"

print("Config ready.")

## 2. Load corpus

In [ ]:
corpus_lines = [
    l.strip() for l in Path(CORPUS_PATH).read_text(encoding="utf-8").splitlines()
    if l.strip() and len(l.split()) >= 4
]

print(f"Loaded {len(corpus_lines):,} lines from {CORPUS_PATH}")
print("\nFirst 5 lines:")
for l in corpus_lines[:5]:
    print(" ", l)

## 3. Prompt & API call

In [ ]:
SYSTEM_PROMPT = """\
You are an expert in Shakespearean English. Your task is to expand a compressed poetic line
into natural archaic prose — the kind of English Shakespeare himself would write.

STRICT RULES:
1. Every word from the original line MUST appear verbatim somewhere in EACH expansion.
2. Preserve archaic register: use thee, thou, thy, doth, hath, art, wilt, dost, etc.
   Do NOT modernise into 'you', 'your', 'does', 'has'.
3. You may add new archaic words and reorder freely to make prose read naturally.
4. Each expansion must be meaningfully different from the others (different word order,
   different added words — not just minor punctuation changes).
5. Return ONLY a JSON array of exactly {n} strings. No explanation, no markdown fences.
"""

USER_TEMPLATE = """\
Expand this Shakespeare line into {n} different archaic prose versions:

{line}
"""


def call_api(line: str, n: int = N_EXPANSIONS) -> list[str] | None:
    """Call Claude and return a list of n expansions, or None on failure."""
    for attempt in range(RETRY_LIMIT):
        try:
            response = client.messages.create(
                model=MODEL,
                max_tokens=MAX_TOKENS,
                system=SYSTEM_PROMPT.format(n=n),
                messages=[{"role": "user", "content": USER_TEMPLATE.format(line=line, n=n)}],
            )
            raw = response.content[0].text.strip()
            # Strip markdown fences if model adds them anyway
            raw = re.sub(r"^```json\s*|^```\s*|\s*```$", "", raw, flags=re.MULTILINE).strip()
            expansions = json.loads(raw)
            if isinstance(expansions, list) and len(expansions) == n:
                return [str(e).strip() for e in expansions]
            print(f"  Bad response length ({len(expansions)} != {n}), retrying...")
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)
    return None


# Smoke test
test = call_api(corpus_lines[0])
print("Smoke test on:", corpus_lines[0])
print()
if test:
    for i, exp in enumerate(test, 1):
        print(f"  {i}. {exp}")
else:
    print("FAILED — check your API key.")

## 4. Validation

In [ ]:
def tokenize(text: str) -> list[str]:
    """Lowercase, strip punctuation, split into tokens."""
    return re.findall(r"[a-z']+", text.lower())


def validate(original: str, expanded: str) -> tuple[bool, list[str]]:
    """
    Check every token in original appears in expanded.
    Returns (is_valid, missing_tokens).
    """
    orig_tokens = tokenize(original)
    exp_tokens  = set(tokenize(expanded))
    missing     = [t for t in orig_tokens if t not in exp_tokens]
    return len(missing) == 0, missing


# Test validation
if test:
    print("Validation results for smoke test:")
    for i, exp in enumerate(test, 1):
        ok, missing = validate(corpus_lines[0], exp)
        print(f"  {i}. {'✓' if ok else '✗'} {'' if ok else f'Missing: {missing}'}")
        print(f"     {exp}")

## 5. Generate full dataset

In [ ]:
# ── Resume from checkpoint if it exists ───────────────────────────────────────
checkpoint_path = Path(CHECKPOINT)
if checkpoint_path.exists():
    records = json.loads(checkpoint_path.read_text())
    done    = {r["target"] for r in records}
    remaining = [l for l in corpus_lines if l not in done]
    print(f"Resuming: {len(records):,} done, {len(remaining):,} remaining.")
else:
    records   = []
    remaining = corpus_lines
    print(f"Starting fresh: {len(remaining):,} lines to process.")

failures = []

# ── Main loop ─────────────────────────────────────────────────────────────────
for line in tqdm(remaining, desc="Expanding lines"):

    expansions = None
    # Allow a second full attempt if validation fails on first
    for attempt in range(2):
        result = call_api(line)
        if result is None:
            break
        # Validate all expansions
        all_valid = True
        for exp in result:
            ok, missing = validate(line, exp)
            if not ok:
                all_valid = False
                break
        if all_valid:
            expansions = result
            break
        time.sleep(SLEEP_BETWEEN)

    if expansions:
        for exp in expansions:
            records.append({"input": exp, "target": line})
    else:
        failures.append(line)

    # Save checkpoint after every line
    checkpoint_path.write_text(json.dumps(records, indent=2))
    time.sleep(SLEEP_BETWEEN)

print(f"\nDone. {len(records):,} pairs generated, {len(failures)} lines failed.")

## 6. Inspect results

In [ ]:
import random

df = pd.DataFrame(records)
print(f"Total training pairs : {len(df):,}")
print(f"Unique target lines  : {df['target'].nunique():,}")
print(f"Expansions per line  : {len(df) / df['target'].nunique():.1f} avg")
print()

df["input_words"]  = df["input"].str.split().str.len()
df["target_words"] = df["target"].str.split().str.len()
print(df[["input_words", "target_words"]].describe().round(1))

print("\n=== Random sample ===")
prev_target = None
for _, row in df.sample(9, random_state=42).iterrows():
    if row["target"] != prev_target:
        print(f"\nTARGET : {row['target']}")
        prev_target = row["target"]
    print(f"  INPUT: {row['input']}")

## 7. Failures review

In [ ]:
if failures:
    print(f"{len(failures)} lines failed validation after all retries:")
    for f in failures:
        print(" ", f)
    Path(FAILURES_LOG).write_text(json.dumps(failures, indent=2))
    print(f"\nSaved to {FAILURES_LOG} for manual review.")
else:
    print("No failures — all lines passed validation.")

## 8. Export

In [ ]:
# CSV
df[["input", "target"]].to_csv(OUTPUT_CSV, index=False)
print(f"Saved {OUTPUT_CSV}  ({len(df):,} rows)")

# JSONL
with open(OUTPUT_JSONL, "w") as f:
    for _, row in df.iterrows():
        f.write(json.dumps({"input": row["input"], "target": row["target"]}) + "\n")
print(f"Saved {OUTPUT_JSONL} ({len(df):,} rows)")

# Clean up checkpoint
if checkpoint_path.exists():
    checkpoint_path.unlink()
    print("Checkpoint removed.")

print(f"\nDone. {len(df):,} training pairs ready.")

## 9. (Optional) Train / val / test split

In [ ]:
from sklearn.model_selection import train_test_split

# Split on unique target lines so no target leaks across splits
unique_targets = df["target"].unique()
train_t, temp_t = train_test_split(unique_targets, test_size=0.2, random_state=42)
val_t,   test_t = train_test_split(temp_t,         test_size=0.5, random_state=42)

train_df = df[df["target"].isin(train_t)]
val_df   = df[df["target"].isin(val_t)]
test_df  = df[df["target"].isin(test_t)]

train_df.to_csv("train.csv", index=False)
val_df.to_csv("val.csv",     index=False)
test_df.to_csv("test.csv",   index=False)

print(f"Train : {len(train_df):,} pairs ({len(train_t):,} unique targets)")
print(f"Val   : {len(val_df):,} pairs  ({len(val_t):,} unique targets)")
print(f"Test  : {len(test_df):,} pairs  ({len(test_t):,} unique targets)")